# 🖥️ Módulo 07: Sistemas, Inferencia Eficiente y Cuantización
## Capítulo 4: Inferencia Local de Alto Rendimiento: Exportación a ONNX Runtime, Formato GGUF, llama.cpp y Aceleración en CPU/GPU Integrada

> *"La culminación de todo AI Researcher e Ingeniero de Sistemas no es solo saber derivar un gradiente o entrenar un Transformer, sino saber llevarlo al mundo real: ejecutar modelos de vanguardia de forma soberana, privada y ultrarrápida en el silicio disponible (desde una CPU x86 con AVX-512 o ARM NEON hasta una GPU integrada Radeon 780M o un clúster de servidores). En este capítulo final exploramos la ingeniería de grafos computacionales con ONNX Runtime, el revolucionario formato binario GGUF con mapeo de memoria `mmap`, el ecosistema de Georgi Gerganov (`llama.cpp`) y la orquestación de inferencia local sin dependencias en la nube."*

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mcarbonell/algo-to-ai/blob/main/notebooks/07_systems_and_efficiency/04_local_inference_and_onnx.ipynb)

---

### ⚙️ Inicialización del Entorno
Cargamos las librerías matemáticas, ONNX Runtime y fijamos semillas para asegurar reproducibilidad determinista.

In [18]:
# !pip install -q numpy matplotlib torch onnx onnxruntime
from typing import Tuple, List, Dict, Optional, Any
import struct
import mmap
import os
import time
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

try:
    import onnx
    import onnxruntime as ort
    has_onnx = True
except ImportError:
    has_onnx = False

np.random.seed(42)
torch.manual_seed(42)
print(f"✅ Entorno listo | ONNX Runtime disponible: {has_onnx}")

✅ Entorno listo | ONNX Runtime disponible: True


---

## 1. 📜 Contexto Histórico y Proceso de Descubrimiento

### El Desafío de Producción: Superar el Intérprete de Python
PyTorch es la herramienta indiscutible de investigación por su autodiferenciación dinámica y flexibilidad en Python.
Sin embargo, desplegar modelos en Python acarrea fricciones operativas:
* **Sobrecoste de despacho (Dispatch Overhead):** El intérprete de Python añade microsegundos de sobrecoste por cada llamada a kernel.
* **Dependencias masivas:** Entornos de ejecución de varios gigabytes (CUDA, PyTorch, librerías compartidas).
* **Falta de portabilidad:** Dificultad para ejecutar en teléfonos, microcontroladores o aplicaciones de escritorio ligeras.

### ONNX (Open Neural Network Exchange - 2017) & ONNX Runtime (Microsoft)
Creado conjuntamente por Facebook y Microsoft, **ONNX** definió un estándar abierto para representar grafos de cómputo en formato estático binario basado en Protocol Buffers:
1. **Fusión de Operadores (*Graph Optimization*):** Identifica patrones en el grafo como `MatMul + BiasAdd + ReLU` y los consolida en una sola operación optimizada (`Gemm`), evitando lecturas/escrituras intermedias a memoria.
2. **Execution Providers (EPs):** Una abstracción unificada que permite compilar el mismo archivo `.onnx` para:
   * **CPU:** Instrucciones vectoriales AVX2, AVX-512 y ARM NEON.
   * **GPU Integrada / Windows:** `DirectMLExecutionProvider` (DirectX 12 / DirectML, compatible con GPUs AMD Radeon 780M e Intel Xe).
   * **GPU Servidor:** `CUDAExecutionProvider` y `TensorRTExecutionProvider`.

### La Revolución de `llama.cpp` y el Formato GGUF (Georgi Gerganov, 2023)
En marzo de 2023, el ingeniero búlgaro **Georgi Gerganov** creó `llama.cpp`, una reimplementación del modelo LLaMA en C/C++ puro con cero dependencias externas.
En agosto de 2023, el ecosistema introdujo el formato binario definitivo: **GGUF (GPT-Generated Unified Format)**.

**La Clave del Rendimiento de GGUF: Memory-Mapped I/O (`mmap`)**
* En los formatos clásicos (como `pickle` o `safetensors` ingenuo), el proceso lee el archivo mediante llamadas `read()`, duplicando los gigabytes en RAM.
* GGUF estructura todos los tensores alineados en múltiplos de 32 bytes. Mediante `mmap()`, el sistema operativo asigna los datos de disco directamente al espacio de memoria virtual del proceso.
* **Resultado:** Cargar un modelo de 70B tarda **fracciones de segundo**, y la memoria se comparte instantáneamente entre múltiples procesos sin sobrecargar la memoria RAM.

---

## 2. 🧠 Intuición Geométrica y Mecánica (Mentalidad de Algoritmista)

### Anatomía Binaria de un Archivo GGUF
Un archivo GGUF es un contenedor binario estricto compuesto por:
```
┌────────────────────────────────────────────────────────┐
│ CABECERA (Header):                                     │
│   - Magic Number: 0x46554747 ('GGUF' en Little-Endian) │
│   - Versión (uint32, ej: 3)                            │
│   - Tensor Count (uint64)                              │
│   - Metadata KV Count (uint64)                         │
├────────────────────────────────────────────────────────┤
│ METADATOS CLAVE-VALOR (Metadata KV Pairs):             │
│   - "general.architecture": "llama"                    │
│   - "llama.context_length": 4096                       │
│   - "tokenizer.ggml.tokens": ["<s>", "hello", ...]     │
├────────────────────────────────────────────────────────┤
│ ÍNDICE DE TENSORES (Tensor Info Array):                │
│   - Nombre (string)                                    │
│   - Dimensiones (uint32[])                             │
│   - Tipo Cuantización (Q4_K, Q8_0, FP16...)            │
│   - Offset absoluto del tensor en el archivo           │
├────────────────────────────────────────────────────────┤
│ RELLENO DE ALINEACIÓN (Padding a 32 bytes)             │
├────────────────────────────────────────────────────────┤
│ BLOBS BINARIOS DE DATOS DE TENSORES (Mapeables mmap)   │
└────────────────────────────────────────────────────────┘
```

### La Ventaja Algorítmica de `mmap`:
Al usar `mmap`, el puntero de memoria en C++ / Python apunta a los bytes del SSD NVMe a través del Page Cache del kernel. Si la memoria física escasea, el kernel simplemente purga las páginas no modificadas sin escribir en swap, reduciendo drásticamente la presión sobre la RAM.

---

## 3. 🛠️ Implementación "From Scratch" (Primeros Principios)

### Parte 1: Parser y Serializador Binario Tipo GGUF From Scratch
Construyamos una clase en Python puro que serialice tensores en un archivo binario estructurado con magic number `GGUF`, cabecera, metadatos, tabla de offsets y alineación a 32 bytes, y luego lo lea instantáneamente mediante `mmap`:

In [19]:
class MiniGGUFWriter:
    """
    Serializador de archivos binarios tipo GGUF con alineación de memoria para mmap.
    """
    MAGIC = b"GGUF"
    VERSION = 3
    ALIGNMENT = 32
    
    def __init__(self, filepath: str):
        self.filepath = filepath
        self.metadata = {}
        self.tensors = {}
        
    def add_metadata(self, key: str, value: str):
        self.metadata[key] = value
        
    def add_tensor(self, name: str, array: np.ndarray):
        self.tensors[name] = array.astype(np.float32)
        
    def write(self):
        with open(self.filepath, "wb") as f:
            # 1. Cabecera
            f.write(self.MAGIC)
            f.write(struct.pack("<I", self.VERSION))
            f.write(struct.pack("<Q", len(self.tensors)))
            f.write(struct.pack("<Q", len(self.metadata)))
            
            # 2. Metadatos
            for k, v in self.metadata.items():
                k_bytes = k.encode("utf-8")
                v_bytes = v.encode("utf-8")
                f.write(struct.pack("<Q", len(k_bytes)) + k_bytes)
                f.write(struct.pack("<Q", len(v_bytes)) + v_bytes)
                
            # 3. Informacion de Tensores (Calculo de offsets)
            tensor_info_pos = f.tell()
            # Espacio reservado para informacion y alineacion
            tensor_data_offsets = {}
            
            # Primer calculo de donde comenzaran los datos
            header_size = f.tell()
            for name, arr in self.tensors.items():
                n_bytes = name.encode("utf-8")
                header_size += 8 + len(n_bytes) + 4 + (4 * len(arr.shape)) + 8
                
            # Alineacion al siguiente multiplo de 32 bytes
            curr_offset = ((header_size + self.ALIGNMENT - 1) // self.ALIGNMENT) * self.ALIGNMENT
            
            for name, arr in self.tensors.items():
                n_bytes = name.encode("utf-8")
                f.write(struct.pack("<Q", len(n_bytes)) + n_bytes)
                f.write(struct.pack("<I", len(arr.shape)))
                for d in arr.shape:
                    f.write(struct.pack("<I", d))
                f.write(struct.pack("<Q", curr_offset))
                tensor_data_offsets[name] = curr_offset
                curr_offset += arr.nbytes
                
            # 4. Rellenar padding hasta el inicio del primer tensor
            pad_size = tensor_data_offsets[list(self.tensors.keys())[0]] - f.tell()
            f.write(b"\x00" * pad_size)
            
            # 5. Escribir blobs binarios contiguos de los tensores
            for name, arr in self.tensors.items():
                f.write(arr.tobytes())


class MiniGGUFReader:
    """
    Lector de archivos GGUF usando mapeo directo de memoria (mmap).
    """
    def __init__(self, filepath: str):
        self.filepath = filepath
        self.f = open(filepath, "rb")
        self.mm = mmap.mmap(self.f.fileno(), 0, access=mmap.ACCESS_READ)
        self.metadata = {}
        self.tensor_info = {}
        self._parse()
        
    def _parse(self):
        magic = self.mm[:4]
        assert magic == b"GGUF", f"Formato invalido: {magic}"
        version, n_tensors, n_meta = struct.unpack_from("<IQQ", self.mm, 4)
        pos = 24
        
        # Leer metadatos
        for _ in range(n_meta):
            k_len = struct.unpack_from("<Q", self.mm, pos)[0]
            pos += 8
            key = self.mm[pos:pos+k_len].decode("utf-8")
            pos += k_len
            v_len = struct.unpack_from("<Q", self.mm, pos)[0]
            pos += 8
            val = self.mm[pos:pos+v_len].decode("utf-8")
            pos += v_len
            self.metadata[key] = val
            
        # Leer tabla de tensores
        for _ in range(n_tensors):
            n_len = struct.unpack_from("<Q", self.mm, pos)[0]
            pos += 8
            name = self.mm[pos:pos+n_len].decode("utf-8")
            pos += n_len
            n_dims = struct.unpack_from("<I", self.mm, pos)[0]
            pos += 4
            shape = struct.unpack_from(f"<{n_dims}I", self.mm, pos)
            pos += 4 * n_dims
            offset = struct.unpack_from("<Q", self.mm, pos)[0]
            pos += 8
            self.tensor_info[name] = (shape, offset)
            
    def get_tensor(self, name: str) -> np.ndarray:
        """
        Devuelve el tensor mapeado en memoria (Zero-Copy).
        """
        shape, offset = self.tensor_info[name]
        total_elems = int(np.prod(shape))
        # Crear un ndarray que apunta directamente al buffer de memoria mmap
        return np.ndarray(shape, dtype=np.float32, buffer=self.mm, offset=offset)
        
    def close(self):
        self.mm.close()
        self.f.close()

print("✅ Clases MiniGGUFWriter y MiniGGUFReader compiladas exitosamente")

✅ Clases MiniGGUFWriter y MiniGGUFReader compiladas exitosamente


### Demostración Práctica: Serialización GGUF y Carga Zero-Copy con `mmap`
Probemos a escribir y recuperar tensores verificando que el tiempo de acceso es prácticamente instantáneo:

In [20]:
temp_gguf_path = "test_model.gguf"

# 1. Crear y escribir archivo GGUF
writer = MiniGGUFWriter(temp_gguf_path)
writer.add_metadata("architecture", "nanollama")
writer.add_metadata("author", "Algoritmista AI")

w_embed = np.random.randn(256, 64).astype(np.float32)
w_attn = np.random.randn(64, 64).astype(np.float32)
writer.add_tensor("model.embed_tokens.weight", w_embed)
writer.add_tensor("model.layers.0.attn.weight", w_attn)
writer.write()

# 2. Leer con mmap
reader = MiniGGUFReader(temp_gguf_path)
print("Metadatos GGUF leídos:", reader.metadata)
print("Tensores en índice:", list(reader.tensor_info.keys()))

tensor_loaded = reader.get_tensor("model.embed_tokens.weight")
assert np.allclose(w_embed, tensor_loaded)
print(f"Tensor cargado con mmap: forma {tensor_loaded.shape} | dtype {tensor_loaded.dtype}")

reader.close()
if os.path.exists(temp_gguf_path):
    os.remove(temp_gguf_path)
print("🚀 ¡Demostración de lectura GGUF Zero-Copy con mmap completada con éxito!")

Metadatos GGUF leídos: {'architecture': 'nanollama', 'author': 'Algoritmista AI'}
Tensores en índice: ['model.embed_tokens.weight', 'model.layers.0.attn.weight']
Tensor cargado con mmap: forma (256, 64) | dtype float32
🚀 ¡Demostración de lectura GGUF Zero-Copy con mmap completada con éxito!


---

## 4. ⚡ Transición a PyTorch Moderno y ONNX Runtime

Exportemos un modelo pequeño de PyTorch a un grafo estático `.onnx` y ejecutemos la inferencia comparativa con **ONNX Runtime**:

In [21]:
import warnings

class SimpleMLP(nn.Module):
    def __init__(self, in_dim: int = 16, hidden: int = 32, out_dim: int = 4):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden),
            nn.SiLU(),
            nn.Linear(hidden, out_dim)
        )
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)

mlp_model = SimpleMLP()
mlp_model.eval()
dummy_input = torch.randn(1, 16)

# Salida PyTorch
with torch.no_grad():
    pytorch_out = mlp_model(dummy_input).numpy()

onnx_path = "simple_mlp.onnx"
onnx_exported = False

# Exportación a ONNX silenciando el warning de deprecación del exportador clásico
try:
    with warnings.catch_warnings():
        warnings.filterwarnings("ignore", category=DeprecationWarning)
        torch.onnx.export(
            mlp_model,
            dummy_input,
            onnx_path,
            input_names=["input"],
            output_names=["output"],
            dynamic_axes={"input": {0: "batch_size"}, "output": {0: "batch_size"}},
            opset_version=14,
            dynamo=False
        )
    print(f"✅ Modelo exportado a formato ONNX en: {onnx_path}")
    onnx_exported = True
except Exception as e:
    print(f"⚠️ Nota sobre exportador ONNX ({e})")

if onnx_exported and has_onnx:
    # Cargar sesion de inferencia en ONNX Runtime
    session = ort.InferenceSession(onnx_path, providers=["CPUExecutionProvider"])
    ort_inputs = {"input": dummy_input.numpy()}
    ort_out = session.run(None, ort_inputs)[0]
    
    discrepancy = np.max(np.abs(pytorch_out - ort_out))
    print(f"Discrepancia PyTorch vs ONNX Runtime: {discrepancy:.2e}")
    assert np.allclose(pytorch_out, ort_out, atol=1e-5)
    print("🚀 ¡Inferencia con ONNX Runtime idéntica a PyTorch sin necesidad de su runtime!")
elif not has_onnx:
    print("ℹ️ onnxruntime no instalado en el entorno local; exportación ONNX verificada")

if os.path.exists(onnx_path):
    os.remove(onnx_path)


✅ Modelo exportado a formato ONNX en: simple_mlp.onnx
Discrepancia PyTorch vs ONNX Runtime: 2.98e-08
🚀 ¡Inferencia con ONNX Runtime idéntica a PyTorch sin necesidad de su runtime!


### Aceleración en Hardware Específico:
* **En Windows con GPU Integrada (AMD Radeon 780M / Intel Iris Xe):**
  ```python
  # DirectMLExecutionProvider utiliza DirectX 12 para acelerar modelos en cualquier GPU de Windows sin requerir CUDA
  session = ort.InferenceSession("model.onnx", providers=["DmlExecutionProvider"])
  ```
* **En llama.cpp:**
  ```bash
  # Descargar y compilar con soporte para CPU optimizada o Vulkan/DirectML:
  llama-cli -m llama-3-8b-instruct.Q4_K_M.gguf -p "Explícame el modelo Roofline" -n 256
  ```

---

## 5. 🎯 Retos & Experimentos ("Tinker Time")

### Reto 1: Verificación de la Fusión de Operadores
Un optimizador de grafos como ONNX fusiona $W \cdot X + b$ en un único operador `Gemm`. Comprobemos la reducción analítica de accesos a memoria intermedia al fusionar operadores:

In [22]:
# Comparativa analitica de lecturas/escrituras en memoria
def memory_traffic_unfused_vs_fused(N: int, D: int) -> Tuple[int, int]:
    # Sin fusion:
    # 1. MatMul: Lee X (N*D), W (D*D) -> Escribe Y_tmp (N*D)
    # 2. Add: Lee Y_tmp (N*D), B (D) -> Escribe Y_out (N*D)
    # Total bytes transferidos = 4*N*D + D*D + D
    unfused_bytes = (4 * N * D + D * D + D) * 4  # FP32
    
    # Con fusion (Gemm):
    # Lee X (N*D), W (D*D), B (D) -> Escribe directamente Y_out (N*D)
    # Total bytes transferidos = 2*N*D + D*D + D
    fused_bytes = (2 * N * D + D * D + D) * 4
    return unfused_bytes, fused_bytes

unfused_b, fused_b = memory_traffic_unfused_vs_fused(N=128, D=4096)
print(f"Tráfico en bus sin fusión: {unfused_b / 1e6:.2f} MB")
print(f"Tráfico en bus con fusión: {fused_b / 1e6:.2f} MB")
print(f"Ahorro directo de tráfico de memoria: {(1.0 - fused_b / unfused_b) * 100:.1f}%")
assert fused_b < unfused_b
print("✅ La fusión de operadores ahorra ancho de banda al eliminar buffers temporales intermediarios")

Tráfico en bus sin fusión: 75.51 MB
Tráfico en bus con fusión: 71.32 MB
Ahorro directo de tráfico de memoria: 5.6%
✅ La fusión de operadores ahorra ancho de banda al eliminar buffers temporales intermediarios


### Reto 2 (Para resolver): Implementar un Generador Streaming Local
Los clientes web y CLI esperan recibir los tokens en tiempo real (*Server-Sent Events / streaming*) en lugar de esperar a que termine toda la generación.

Implementa a continuación la función generadora `stream_local_tokens(model_fn, initial_tokens, max_new_tokens=10)`:

In [23]:
# TU CÓDIGO DEL RETO 2 AQUÍ
def stream_local_tokens(model_fn, initial_tokens: List[int], max_new_tokens: int = 10):
    """
    Generador en streaming token a token para interfaces interactivas locales.
    """
    # Tu implementación aquí con yield
    pass

---

## 6. 📚 Referencias Fundamentales & Lecturas Recomendadas

### 📄 Proyectos y Papers Seminales
1. **Gerganov, G. (2023):** *"llama.cpp: Inference of Meta's LLaMA model in pure C/C++"*. [GitHub ggerganov/llama.cpp](https://github.com/ggerganov/llama.cpp)
   * *¿Qué estudiar?* La arquitectura modular de GGML y la gestión de tensores en memoria continua.
2. **GGUF Specification (2023):** *"GGUF format description and specifications"*. [Especificación oficial](https://github.com/ggerganov/ggml/blob/master/docs/gguf.md)
   * *¿Qué estudiar?* El estándar binario universal que unificó la inferencia local en la comunidad de código abierto.
3. **Microsoft ONNX Runtime Team (2024):** *"ONNX Runtime: Cross-Platform Performance Accelerator"*. [onnxruntime.ai](https://onnxruntime.ai/)
   * *¿Qué estudiar?* Fusión de operadores y adaptadores de hardware (DirectML, TensorRT, CoreML).
4. **Williams, S., et al. (2009):** *"Roofline: An Insightful Visual Performance Model for Multicore Architectures"*.